# Capstone

## Race Level

### 0. Setup

In [1]:
# ── Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')

# ── Dependency Guard ───────────────────────────────────────────────
# Ensures all required packages are installed in the active kernel.
# Safe to re-run: pip will skip already-installed packages.

import importlib, subprocess, sys

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'fastf1': 'fastf1',
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed ✓')

# ── Library Imports ───────────────────────────────────────────────
import os                        # Working directory checks
import subprocess                # Git command checks
import importlib                 # Runtime dependency checks
import numpy as np               # Numeric support
import pandas as pd              # Tables and diagnostics
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns            # Statistical plotting
import fastf1                    # Formula 1 data access

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier  # Tree-based ensemble

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, brier_score_loss,
)

try:
    from xgboost import XGBClassifier
    print("XGBoost available ✓")
except ImportError:
    XGBClassifier = None
    print("⚠ XGBoost not installed. Run: pip install xgboost")


print(f'fastf1  : {fastf1.__version__}')
print(f'pandas  : {pd.__version__}')

Python  : 3.12.3
NumPy   : 2.4.2
Seed    : 414
All required packages already installed ✓
XGBoost available ✓
fastf1  : 3.8.1
pandas  : 2.3.3


Dataset

In [2]:
DATA_PATH = "f1_strategy_lap_level.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns from {DATA_PATH}")
df.head(3)

Loaded 119,908 rows and 30 columns from f1_strategy_lap_level.csv


,season,round,circuit,event_date,driver_id,Driver,DriverNumber,Team,LapNumber,Stint,...,TrackTemp,Humidity,current_rainfall,track_status,is_safety_car_lap,is_vsc_lap,pit_in_flag,pit_out_flag,total_laps_in_race,lap_in_race_pct
0,2019,1,Australian Grand Prix,2019-03-17,albon,ALB,23,RB,2,1,...,43.5,69.7,False,green,False,False,0,0,58,0.034483
1,2019,1,Australian Grand Prix,2019-03-17,albon,ALB,23,RB,3,1,...,43.4,70.5,False,green,False,False,0,0,58,0.051724
2,2019,1,Australian Grand Prix,2019-03-17,albon,ALB,23,RB,4,1,...,43.4,70.3,False,green,False,False,0,0,58,0.068966


In [3]:
df.columns

Index(['season', 'round', 'circuit', 'event_date', 'driver_id', 'Driver',
       'DriverNumber', 'Team', 'LapNumber', 'Stint', 'Compound', 'TyreLife',
       'stint_lap_number', 'stint_length_observed', 'stint_pct',
       'lap_time_seconds', 'lap_time_delta', 'Position', 'position_change_lap',
       'AirTemp', 'TrackTemp', 'Humidity', 'current_rainfall', 'track_status',
       'is_safety_car_lap', 'is_vsc_lap', 'pit_in_flag', 'pit_out_flag',
       'total_laps_in_race', 'lap_in_race_pct'],
      dtype='object')

Creating necesary features

In [4]:
# ── Feature Engineering ────────────────────────────────────────────
# Create: grid_position, constructor_avg_finish_pos_5race_rolling

# 1. Create grid_position from DriverNumber (grid position is assigned before the race)
# Note: In lap-level data, grid position is constant per driver per race
df['grid_position'] = df.groupby(['season', 'round', 'Driver'])['DriverNumber'].transform('first')

print("✓ grid_position created")
print(f"  dtype: {df['grid_position'].dtype}, missing: {df['grid_position'].isna().sum()}")

# 2. Create constructor_avg_finish_pos_5race_rolling
# Aggregate to race level: for each team in each race, get their average finishing position
# (need to aggregate lap-level data to race level first)
race_level = df.groupby(['season', 'round', 'Team']).agg({
    'Position': 'mean'  # avg finishing position for this team in this race
}).reset_index()
race_level.columns = ['season', 'round', 'Team', 'avg_finish_pos']

# Sort chronologically to enable rolling window
race_level = race_level.sort_values(['Team', 'season', 'round']).reset_index(drop=True)

# Calculate 5-race rolling average per team
race_level['constructor_avg_finish_pos_5race_rolling'] = (
    race_level.groupby('Team')['avg_finish_pos']
    .rolling(window=5, min_periods=1)
    .mean()
    .reset_index(drop=True)
)

# Merge back to original dataframe
df = df.merge(
    race_level[['season', 'round', 'Team', 'constructor_avg_finish_pos_5race_rolling']],
    on=['season', 'round', 'Team'],
    how='left'
)

print("✓ constructor_avg_finish_pos_5race_rolling created")
print(f"  dtype: {df['constructor_avg_finish_pos_5race_rolling'].dtype}, missing: {df['constructor_avg_finish_pos_5race_rolling'].isna().sum()}")

# 3. Summary
print("\n" + "="*70)
print("Features Ready for Logistic Regression:")
print("="*70)
print(f"grid_position shape: {df['grid_position'].shape}")
print(f"constructor_avg_finish_pos_5race_rolling shape: {df['constructor_avg_finish_pos_5race_rolling'].shape}")
print(f"\nTotal dataset shape: {df.shape}")
print(f"Features: ['grid_position', 'constructor_avg_finish_pos_5race_rolling']")

✓ grid_position created
  dtype: int64, missing: 0
✓ constructor_avg_finish_pos_5race_rolling created
  dtype: float64, missing: 0

Features Ready for Logistic Regression:
grid_position shape: (119908,)
constructor_avg_finish_pos_5race_rolling shape: (119908,)

Total dataset shape: (119908, 32)
Features: ['grid_position', 'constructor_avg_finish_pos_5race_rolling']


In [5]:
df.columns

Index(['season', 'round', 'circuit', 'event_date', 'driver_id', 'Driver',
       'DriverNumber', 'Team', 'LapNumber', 'Stint', 'Compound', 'TyreLife',
       'stint_lap_number', 'stint_length_observed', 'stint_pct',
       'lap_time_seconds', 'lap_time_delta', 'Position', 'position_change_lap',
       'AirTemp', 'TrackTemp', 'Humidity', 'current_rainfall', 'track_status',
       'is_safety_car_lap', 'is_vsc_lap', 'pit_in_flag', 'pit_out_flag',
       'total_laps_in_race', 'lap_in_race_pct', 'grid_position',
       'constructor_avg_finish_pos_5race_rolling'],
      dtype='object')

### 1. Target and splitting

In [6]:
TARGET = "top10"
df[TARGET] = ((df['Position'] <= 10) & (df['Position'] >0)).astype(int)
print(df[TARGET].value_counts(dropna=False))

top10
1    65708
0    54200
Name: count, dtype: int64


Splitting

In [7]:
# Temporal split: train (<2022), validation (2022), test (>=2023)
train = df[df["season"] <= 2021].copy()
val   = df[df["season"] == 2022].copy()
test  = df[df["season"] >= 2023].copy()

### 2. Model

In [8]:
# Features: adapt to dataset and remove leakage features
# Context columns (meta): useful for grouping or debugging, not modelled directly
CONTEXT_COLUMNS = ['season', 'round', 'circuit', 'event_date', 'Team', 'Driver']

# Numeric features available pre-race (or computable from past races)
NUMERIC_FEATURES = [
    'grid_position',
    'constructor_avg_finish_pos_5race_rolling',
    'DriverNumber',
    'total_laps_in_race',
]

# Categorical features available pre-race
CATEGORICAL_FEATURES = [
    'circuit',
    'Team',
    # 'Compound' is a strategy choice (hypothetical) — include only if using scenario-level inputs
]

# Columns that leak post-race or are in-race signals we must NOT use for pre-race prediction
LEAKAGE_COLS = [
    'Position', 'position_change_lap', 'LapNumber', 'stint_lap_number',
    'stint_length_observed', 'lap_time_seconds', 'lap_time_delta', 'TyreLife',
    'stint_pct', 'is_safety_car_lap', 'is_vsc_lap', 'pit_in_flag', 'pit_out_flag',
    'lap_in_race_pct'
]

# Drop leakage columns from the dataframe if present
dropped = [c for c in LEAKAGE_COLS if c in df.columns]
if dropped:
    df.drop(columns=dropped, inplace=True)
    print(f"Dropped leakage columns: {dropped}")
else:
    print("No leakage columns found to drop")

# Recompute temporal splits in case df changed
train = df[df['season'] <= 2021].copy()
val   = df[df['season'] == 2022].copy()
test  = df[df['season'] >= 2023].copy()

# Build feature lists from what actually exists in the training set
available_num = [c for c in NUMERIC_FEATURES if c in train.columns]
available_cat = [c for c in CATEGORICAL_FEATURES if c in train.columns]
MODEL_FEATURES = available_num + available_cat

print("Selected model features:")
print("  numeric:", available_num)
print("  categorical:", available_cat)

# Preprocessor builder
def make_preprocessor():
    transformers = []
    if available_num:
        transformers.append(("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), available_num))
    if available_cat:
        transformers.append(("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), available_cat))
    return ColumnTransformer(transformers)

# End of feature setup
print(f"Train shape: {train.shape}; Val shape: {val.shape}; Test shape: {test.shape}")

Dropped leakage columns: ['Position', 'position_change_lap', 'LapNumber', 'stint_lap_number', 'stint_length_observed', 'lap_time_seconds', 'lap_time_delta', 'TyreLife', 'stint_pct', 'is_safety_car_lap', 'is_vsc_lap', 'pit_in_flag', 'pit_out_flag', 'lap_in_race_pct']
Selected model features:
  numeric: ['grid_position', 'constructor_avg_finish_pos_5race_rolling', 'DriverNumber', 'total_laps_in_race']
  categorical: ['circuit', 'Team']
Train shape: (55635, 19); Val shape: (19622, 19); Test shape: (44651, 19)


In [9]:
#logistic regression
def make_lr_pipeline():
    return Pipeline([
        ("prep", make_preprocessor()),
        ("clf", LogisticRegression(random_state=RANDOM_SEED, max_iter=1000)),
    ])


### 3. Baseline Logistic Regression (Experiment 1)

>Fit a calibrated logistic regression baseline using grid_position and constructor_tier only, then compare it against a model that also includes n_stops.

In [ ]:
#train and validate the baseline


### 4. What if scenarios (Experiment 2)

> Run two concrete what-if scenarios for the same driver/circuit pair: one-stop (M-H, longer final stint) versus two-stop (S-M-M, shorter stints).

Scenario A

Scenario B

### 5. Experiment 3

>"Test whether adding avg_pit_stop_duration_s changes calibrated P(is_top10) enough to matter in the comparison plan."